# 08: Nonlinear Inversion (Scipy and MCMC)

## Learning objectives
- Formulate a nonlinear data model and objective function.
- Estimate parameters with deterministic optimization.
- Compare point estimates to posterior distributions.

## Nonlinear example model
We fit:

$$
y(x) = a\,e^{-x/\tau} + c
$$

Unknown parameters are $a, \tau, c$. The model is nonlinear because $\tau$ appears inside an exponential.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True

rng = np.random.default_rng(22)

In [ ]:
# define our model function
def model(x, a, tau, c):
    return a * np.exp(-x / tau) + c
    #return a * x**2 + tau * x + c

x = np.linspace(0, 10, 50)

# Generate synthetic data
a_true, tau_true, c_true = 2.5, 3.0, 0.4
sigma = 0.12
y_true = model(x, a_true, tau_true, c_true)
y_obs = y_true + rng.normal(0.0, sigma, size=x.size)

fig, ax = plt.subplots()
ax.plot(x, y_obs, 'k.', label='Observed data')
ax.plot(x, y_true, 'k--', label='True model')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Synthetic nonlinear dataset')
ax.legend()
plt.show()

## Part A: Grid Search

A **grid search** evaluates the objective function at every point on a regular mesh in parameter space. It requires no derivatives and is guaranteed to find the global minimum within the chosen grid resolution — but its cost grows exponentially with the number of free parameters (the *curse of dimensionality*).

Here we search the full three-parameter space $(a,\,\tau,\,c)$ on a $60^3$ grid. To visualise the 3-D misfit volume we compute 2-D marginal misfits by taking the **minimum over the third parameter** at each grid point, then display them as colored contour plots in a triangular layout — one panel per parameter pair.

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Full 3-D grid over (a, τ, c)
a_vals   = np.linspace(1.0, 4.5, 200)
tau_vals = np.linspace(1.0, 6.0, 400)
c_vals   = np.linspace(-0.3, 1.1, 200)

# Vectorised: loop over a, broadcast over (τ, c)
TT, CC = np.meshgrid(tau_vals, c_vals, indexing='ij')   # (n_tau, n_c)
misfit_3d = np.zeros((len(a_vals), len(tau_vals), len(c_vals)))
for i, a in enumerate(a_vals):
    y_pred = a * np.exp(-x[:, None, None] / TT[None]) + CC[None]
    #y_pred = a * x[:, None, None]**2 + TT[None] * x[:, None, None] + CC[None]
    misfit_3d[i] = np.sum((y_obs[:, None, None] - y_pred) ** 2, axis=0)

# Grid-search best parameters
idx = np.unravel_index(np.argmin(misfit_3d), misfit_3d.shape)
a_gs, tau_gs, c_gs = a_vals[idx[0]], tau_vals[idx[1]], c_vals[idx[2]]
print(f'Grid search:  a = {a_gs:.3f},  τ = {tau_gs:.3f},  c = {c_gs:.4f}')
print(f'True values:  a = {a_true:.3f},  τ = {tau_true:.3f},  c = {c_true:.4f}')

# 2-D marginal misfits: min over the third parameter
# misfit_3d axes: [a, tau, c]
m_a_tau = np.min(misfit_3d, axis=2).T   # (n_tau, n_a) → contourf(a_vals, tau_vals, ...)
m_a_c   = np.min(misfit_3d, axis=1).T   # (n_c,   n_a) → contourf(a_vals, c_vals,   ...)
m_tau_c = np.min(misfit_3d, axis=0).T   # (n_c,   n_tau) → contourf(tau_vals, c_vals, ...)

# Shared log-scale colour range across all panels
lm = [np.log10(m) for m in (m_a_tau, m_a_c, m_tau_c)]
vmin = min(m.min() for m in lm)
vmax = max(m.max() for m in lm)
levels  = np.linspace(vmin, vmax, 40)
ckw     = dict(levels=np.linspace(vmin, vmax, 10), colors='k', linewidths=0.4, alpha=0.4)

# Triangular layout: 2×2 grid, upper-right panel hidden
fig, axes = plt.subplots(2, 2, figsize=(6, 5))
#axes[0, 1].set_visible(False)

panels = [
    (axes[0, 0], a_vals,   tau_vals, lm[0], '$a$',     r'$\tau$', a_gs,   tau_gs, a_true,   tau_true),
    (axes[1, 0], a_vals,   c_vals,   lm[1], '$a$',     '$c$',     a_gs,   c_gs,   a_true,   c_true),
    (axes[1, 1], tau_vals, c_vals,   lm[2], r'$\tau$', '$c$',     tau_gs, c_gs,   tau_true, c_true),
]

cf = None
for ax, xv, yv, Z, xl, yl, xg, yg, xt, yt in panels:
    cf = ax.contourf(xv, yv, Z, levels=levels, cmap='plasma_r')
    ax.contour(xv, yv, Z, **ckw)
    ax.plot(xg, yg, 'w*', markersize=12, label='Grid min', zorder=5)
    ax.plot(xt, yt, 'w^', markersize=9, markerfacecolor='none',
            markeredgewidth=1.5, label='True', zorder=5)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)

axes[0, 0].legend(fontsize=9, loc='upper right')
cax = inset_axes(axes[0, 1], width='5%', height='100%', loc='center left')
fig.colorbar(cf, cax=cax,
             label=r'$\log_{10}(\mathrm{SSE})$')
axes[0, 1].grid(False)
# Hide the spines (borders)
axes[0, 1].spines['top'].set_visible(False)
axes[0, 1].spines['bottom'].set_visible(False)
axes[0, 1].spines['left'].set_visible(False)
axes[0, 1].spines['right'].set_visible(False)

# Hide the ticks and tick labels
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])

fig.suptitle('Grid search: 2-D marginal misfits (min over third parameter)', fontsize=12)
plt.tight_layout()
plt.show()

## Part B (alternative): Newton's Method with `scipy`

Rather than exhaustively scanning parameter space, a **gradient-based optimizer** starts from an initial guess and iteratively follows the negative gradient of the objective function toward a minimum. This requires only a few gradient evaluations per iteration (no Hessian computation), so the method scales to many parameters. The trade-off: it can converge to a **local** minimum if the misfit surface has multiple valleys.

Newton's method is one of the simplest such optimization algorithms: at each iteration, we approximate the objective function locally with a quadratic, then jump to its minimum. The key is the Hessian matrix (second derivatives). Here we use `scipy.optimize.minimize` with `method='Newton-CG'`, which approximates the Hessian using conjugate gradients—a practical compromise that keeps computation fast while preserving Newton's fast convergence near the optimum.

In [ ]:
from scipy.optimize import minimize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Newton's method optimization
initial_guess_newton = np.array([4.0, 2.0, 1.0])
path = [initial_guess_newton.copy()]

def save_path(params):
    path.append(params.copy())

def objective(params, x, y):
    return np.sum((y - model(x, *params)) ** 2)

def gradient(params, x, y):
    """Compute gradient of SSE with respect to (a, tau, c)"""
    a, tau, c = params
    y_pred = model(x, a, tau, c)
    residual = y - y_pred
    
    # Partial derivatives of model w.r.t. each parameter
    exp_term = np.exp(-x / tau)
    dm_da = exp_term
    dm_dtau = a * exp_term * (x / tau**2)
    dm_dc = np.ones_like(x)
    
    # Chain rule: d(SSE)/d(param) = -2 * sum(residual * dm/d(param))
    grad_a   = -2 * np.sum(residual * dm_da)
    grad_tau = -2 * np.sum(residual * dm_dtau)
    grad_c   = -2 * np.sum(residual * dm_dc)
    
    return np.array([grad_a, grad_tau, grad_c])

result_newton = minimize(
    objective,
    initial_guess_newton,
    args=(x, y_obs),
    method='Newton-CG',
    jac=gradient,
    callback=save_path,
    options={'xtol': 1e-8},
)
p_opt_newton = result_newton.x
path = np.array(path)   # (n_steps, 3)
print(f'Newton-CG estimate [a, τ, c]: {np.round(p_opt_newton, 4)}  ({len(path)} iterations)')

# --- Data fit with Newton's method ---
y_opt_newton = model(x, *p_opt_newton)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y_obs, 'k.', label='Observed')
ax.plot(x, y_true, 'k--', label='True')
ax.plot(x, y_opt_newton, 'g-', label="Newton's fit", linewidth=2)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title("Newton's method nonlinear inversion result")
ax.legend()
plt.show()

# --- Triangular misfit plot with Newton's method path ---
fig, axes = plt.subplots(2, 2, figsize=(6, 5))
axes[0, 1].set_visible(False)

panels = [
    (axes[0, 0], a_vals,   tau_vals, lm[0], '$a$',     r'$\tau$', 0, 1, a_true,   tau_true),
    (axes[1, 0], a_vals,   c_vals,   lm[1], '$a$',     '$c$',     0, 2, a_true,   c_true),
    (axes[1, 1], tau_vals, c_vals,   lm[2], r'$\tau$', '$c$',     1, 2, tau_true, c_true),
]

cf = None
for ax, xv, yv, Z, xl, yl, pi, pj, xt, yt in panels:
    cf = ax.contourf(xv, yv, Z, levels=levels, cmap='plasma_r')
    ax.contour(xv, yv, Z, **ckw)
    ax.plot(xt, yt, 'w^', markersize=9, markerfacecolor='none',
            markeredgewidth=1.5, label='True', zorder=5)
    ax.plot(path[:, pi], path[:, pj], 'g.-', linewidth=1.5, markersize=4,
            label="Newton's path", zorder=10)
    ax.plot(path[0, pi], path[0, pj], 'gs', markersize=7, label='Start',    zorder=11)
    ax.plot(p_opt_newton[pi],   p_opt_newton[pj],   'g*', markersize=11, label='Solution', zorder=11)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)

axes[0, 0].legend(fontsize=8, loc='upper right')
cax = inset_axes(axes[0, 1], width='5%', height='100%', loc='center left')
fig.colorbar(cf, cax=cax,
             label=r'$\log_{10}(\mathrm{SSE})$')
axes[0, 1].grid(False)
axes[0, 1].spines['top'].set_visible(False)
axes[0, 1].spines['bottom'].set_visible(False)
axes[0, 1].spines['left'].set_visible(False)
axes[0, 1].spines['right'].set_visible(False)
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])

fig.suptitle(f"Newton's method on misfit surface ({len(path_newton)} iterations)", fontsize=12)
plt.tight_layout()
plt.show()

## Part B alternative: Deterministic Optimization with `scipy`

Newton's method requires an explicit function that computes the gradient, which adds a lot of work for us up front. There are other methods that don't require this and result in a simpler approach:


**L-BFGS-B** (**Limited-memory Broyden–Fletcher–Goldfarb–Shanno with Bounds**) is an excellent balance between simplicity and efficiency:

- **BFGS** approximates the Hessian (matrix of second derivatives) on-the-fly, giving near-quadratic convergence far from the optimum.
- **Limited-memory** keeps only the last $m$ iterations (typically $m=10$), avoiding $O(n_p^2)$ memory usage.
- **B** means we can impose simple bound constraints on parameters (e.g., $a > 0$, $\tau > 0.2$).

This makes L-BFGS-B practical for 3–100 parameters, with convergence speed between first-order (steepest descent) and full Newton. A `callback` records all three parameter values $(a,\tau,c)$ after each iteration so we can project the solution path onto every panel of the misfit plot from Part A.

In [ ]:
from scipy.optimize import minimize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

initial_guess = np.array([4.0, 2.0, 1.0])
path = [initial_guess.copy()]

result = minimize(
    objective,
    initial_guess,
    args=(x, y_obs),
    method='L-BFGS-B',
    bounds=[(0.0, 10.0), (0.2, 20.0), (-2.0, 5.0)],
    callback=save_path,
    options={'ftol': 1e-14, 'gtol': 1e-8},
)
print(result)
p_opt = result.x
path  = np.array(path)   # (n_steps, 3)
print(f'L-BFGS-B estimate [a, τ, c]: {np.round(p_opt, 4)}  ({len(path)} iterations)')

# --- Data fit ---
y_opt = model(x, *p_opt)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y_obs, 'k.', label='Observed')
ax.plot(x, y_true, 'k--', label='True')
ax.plot(x, y_opt, 'r-', label='Scipy fit')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Deterministic nonlinear inversion result')
ax.legend()
plt.show()

# --- Triangular misfit plot with optimizer path ---
# Reuses misfit contour variables (levels, ckw, lm) computed in the grid-search cell
fig, axes = plt.subplots(2, 2, figsize=(6, 5))
axes[0, 1].set_visible(False)

# (panel axes, x-grid, y-grid, log-misfit, x-label, y-label, path col x, path col y, x_true, y_true)
panels = [
    (axes[0, 0], a_vals,   tau_vals, lm[0], '$a$',     r'$\tau$', 0, 1, a_true,   tau_true),
    (axes[1, 0], a_vals,   c_vals,   lm[1], '$a$',     '$c$',     0, 2, a_true,   c_true),
    (axes[1, 1], tau_vals, c_vals,   lm[2], r'$\tau$', '$c$',     1, 2, tau_true, c_true),
]

cf = None
for ax, xv, yv, Z, xl, yl, pi, pj, xt, yt in panels:
    cf = ax.contourf(xv, yv, Z, levels=levels, cmap='plasma_r')
    ax.contour(xv, yv, Z, **ckw)
    ax.plot(xt, yt, 'w^', markersize=9, markerfacecolor='none',
            markeredgewidth=1.5, label='True', zorder=5)
    ax.plot(path[:, pi], path[:, pj], 'r.-', linewidth=1.5, markersize=4,
            label='Optimizer path', zorder=10)
    ax.plot(path[0, pi], path[0, pj], 'rs', markersize=7, label='Start',    zorder=11)
    ax.plot(p_opt[pi],   p_opt[pj],   'r*', markersize=11, label='Solution', zorder=11)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)

axes[0, 0].legend(fontsize=9, loc='upper right')
cax = inset_axes(axes[0, 1], width='5%', height='100%', loc='center left')
fig.colorbar(cf, cax=cax,
             label=r'$\log_{10}(\mathrm{SSE})$')
axes[0, 1].grid(False)
# Hide the spines (borders)
axes[0, 1].spines['top'].set_visible(False)
axes[0, 1].spines['bottom'].set_visible(False)
axes[0, 1].spines['left'].set_visible(False)
axes[0, 1].spines['right'].set_visible(False)

# Hide the ticks and tick labels
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])

fig.suptitle(f'Optimizer path on misfit surface ({len(path)} iterations)', fontsize=12)
plt.tight_layout()
plt.show()

## Part C: Bayesian Inference with MCMC (emcee)

Deterministic optimization gives a **point estimate** $\mathbf{m}_{\text{opt}}$, but tells us nothing about uncertainty. MCMC (Markov Chain Monte Carlo) samples from the **posterior distribution** $P(\mathbf{m} \mid \mathbf{d})$, giving us not just our best guess, but the full range of plausible parameter values and their correlations.

### The Bayesian Framework

We assume:
- **Forward model**: $\hat{d_i}(\mathbf{m})$ gives the $i$-th predicted data value for model parameters $\mathbf{m}$.
- **Likelihood**: $P(\mathbf{d} \mid \mathbf{m}) = \exp\left(-\frac{1}{2\sigma^2}\sum_i (d_i - \hat{d_i}(\mathbf{m}))^2\right)$ (Gaussian noise with known $\sigma$)
- **Prior**: $P(\mathbf{m})$ — our prior knowledge/constraints (typically, uniform bounds on physical ranges)
- **Posterior**: $P(\mathbf{m} \mid \mathbf{d}) \propto P(\mathbf{d} \mid \mathbf{m}) \cdot P(\mathbf{m})$ (Bayes' rule)

We cannot evaluate the posterior directly (it's unnormalized), but we can sample from it using MCMC.

### Affine Invariant Ensemble Sampler (emcee)

**emcee** is a Python package implementing the **Affine Invariant Ensemble Sampler**, a robust MCMC method that:
- Runs **multiple walkers** in parallel (we use 24), exploring the posterior jointly
- Adapts automatically to the shape of the posterior (hence "affine invariant")
- Requires only the **log-posterior** (unnormalized); derivatives not needed
- Produces a chain of samples: shape `(n_iterations, n_walkers, n_parameters)`

After discarding burn-in samples (first 500 iterations to reach equilibrium), we flatten the remaining chain to get posterior samples we can analyze: marginal distributions, correlations, mean/std, credible intervals.

In [ ]:
# Define log-likelihood, log-prior, and log-posterior functions for MCMC

def log_likelihood(params, x, y, sigma):
    """
    Compute log-likelihood for Gaussian noise model.
    
    Parameters
    ----------
    params : ndarray
        Model parameters [a, tau, c]
    x : ndarray
        Predictor variable
    y : ndarray
        Observed (noisy) data
    sigma : float
        Standard deviation of noise (known)
        
    Returns
    -------
    log_likelihood : float
        Log of Gaussian likelihood = -0.5 * chi-squared
    """
    y_pred = model(x, *params)
    chi_squared = np.sum(((y - y_pred) / sigma) ** 2)
    return -0.5 * chi_squared


def log_prior(params):
    """
    Compute log-prior using uniform (flat) priors with physical bounds.
    
    Parameters
    ----------
    params : ndarray
        Model parameters [a, tau, c]
        
    Returns
    -------
    log_prior : float
        0.0 if all parameters satisfy bounds, -inf otherwise
    """
    a, tau, c = params
    
    # Physical bounds
    if 0.0 < a < 10.0 and 0.2 < tau < 20.0 and -2.0 < c < 5.0:
        return 0.0
    else:
        return -np.inf


def log_posterior(params, x, y, sigma):
    """
    Compute log-posterior = log-prior + log-likelihood.
    
    Parameters
    ----------
    params : ndarray
        Model parameters [a, tau, c]
    x : ndarray
        Predictor variable
    y : ndarray
        Observed data
    sigma : float
        Standard deviation of noise
        
    Returns
    -------
    log_posterior : float
        Log-posterior probability
    """
    prior = log_prior(params)
    
    # If prior is -inf, return immediately (parameter outside bounds)
    if np.isinf(prior):
        return -np.inf
    
    # Otherwise, add log-likelihood
    return prior + log_likelihood(params, x, y, sigma)

In [ ]:
import emcee
import corner

nwalkers = 24
ndim = 3
start = p_opt + 1e-2 * rng.normal(size=(nwalkers, ndim))

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_posterior, args=(x, y_obs, sigma)
)
sampler.run_mcmc(start, 3000, progress=False)

chain = sampler.get_chain(discard=500, flat=True)
p_mean = np.mean(chain, axis=0)
p_std  = np.std(chain, axis=0)

print('MCMC posterior mean [a, τ, c]:', np.round(p_mean, 4))
print('MCMC posterior std  [a, τ, c]:', np.round(p_std, 4))

In [ ]:
fig = corner.corner(
    chain,
    labels=[r'$a$', r'$\tau$', r'$c$'],
    show_titles=True,
    bins=30,
    quantiles=[0.16, 0.5, 0.84],
    truths=[a_true, tau_true, c_true], # adds blue lines at the "known" true values
    use_math_text=True,
)
fig.suptitle('Posterior marginals (emcee)', fontsize=13, y=1.01)
plt.show()

## Exercise (TODO)
1. Try worse initial guesses for deterministic optimization.
2. Increase/decrease `sigma` and observe posterior width changes.
3. Compare deterministic estimate to posterior mean.

## Checkpoint questions
1. Why can nonlinear inversion depend on initial guess?
2. What extra information does MCMC provide beyond one best-fit model?
3. How does data noise level affect posterior uncertainty?

## Common mistakes
- Using priors with impossible parameter ranges.
- Confusing likelihood scale (sigma) with model parameters.
- Running too few MCMC steps and interpreting noisy chains.

## Summary
- You solved a nonlinear inverse problem deterministically.
- You formulated a Bayesian posterior and sampled it (with `emcee` if available).
- You compared point estimates and uncertainty-aware inference.

## Congratulations!
You've reached the end of the notebook! Keep up the great work! 😊